# Sales Prediction using Python
### CodeAlpha Internship — Task 4

Predict future sales based on advertising spend across TV, Radio, and Newspaper platforms.

---
### Table of Contents
1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis
4. Data Cleaning & Outlier Detection
5. Feature Engineering
6. Feature Selection & Splitting
7. Model Training
8. Model Evaluation
9. Visualisations
10. Business Insights & Scenario Analysis
11. Conclusion

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded successfully")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Advertising.csv")
df.drop(columns=[c for c in df.columns if "Unnamed" in c], inplace=True)
print(f"Shape: {df.shape}")
df.head(10)

## 3. Exploratory Data Analysis
### 3.1 Basic Info

In [ ]:
print(df.info())
print("\nMissing Values:\n", df.isnull().sum())

### 3.2 Statistical Summary

In [ ]:
df.describe().round(2)

### 3.3 Distribution Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), df.columns):
    ax.hist(df[col], bins=25, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.axvline(df[col].mean(), color="#C44E52", linestyle="--", linewidth=1.5,
               label=f"Mean={df[col].mean():.1f}")
    ax.set_title(f"{col} Distribution", fontweight="bold")
    ax.legend(fontsize=9)
plt.suptitle("Feature Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="RdYlGn",
            linewidths=0.5, annot_kws={"size": 12})
plt.title("Correlation Matrix", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

### 3.5 Pairplot

In [ ]:
sns.pairplot(df, diag_kind="kde", plot_kws={"alpha": 0.5})
plt.suptitle("Pairplot — Advertising Dataset", y=1.02, fontweight="bold")
plt.show()

## 4. Data Cleaning & Outlier Detection

In [ ]:
df_clean = df.copy()

print(f"{"Feature":<12} {"Outliers":>10}  IQR Range")
print("-" * 45)
for col in df_clean.columns:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n = len(df_clean[(df_clean[col]<lower)|(df_clean[col]>upper)])
    print(f"{col:<12} {n:>10}  [{lower:.1f}, {upper:.1f}]")

df_clean.to_csv("Cleaned_Data.csv", index=False)
print("\nCleaned_Data.csv saved")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col in zip(axes, df_clean.columns):
    ax.boxplot(df_clean[col], patch_artist=True,
               boxprops=dict(facecolor="#4C72B0", alpha=0.7))
    ax.set_title(col, fontweight="bold")
plt.suptitle("Boxplots — Outlier Check", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df_feat = df_clean.copy()
df_feat["TV_Radio_Interaction"] = df_feat["TV"] * df_feat["Radio"]
df_feat["Total_Ad_Spend"]       = df_feat["TV"] + df_feat["Radio"] + df_feat["Newspaper"]
df_feat["TV_Ratio"]             = df_feat["TV"] / df_feat["Total_Ad_Spend"]
df_feat["Digital_Spend"]        = df_feat["Radio"] + df_feat["Newspaper"]
print("New features: TV_Radio_Interaction, Total_Ad_Spend, TV_Ratio, Digital_Spend")
df_feat.head()

## 6. Feature Selection & Train/Test Split

In [ ]:
FEATURES = ["TV", "Radio", "Newspaper"]
TARGET   = "Sales"

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}")

## 7. Model Training

In [ ]:
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge Regression"  : Ridge(alpha=1.0),
    "Lasso Regression"  : Lasso(alpha=0.1),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=100, random_state=42),
    "Polynomial (deg2)" : Pipeline([
                              ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                              ("lr",   LinearRegression())
                          ]),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        "model"  : model,
        "y_pred" : y_pred,
        "rmse"   : np.sqrt(mean_squared_error(y_test, y_pred)),
        "mae"    : mean_absolute_error(y_test, y_pred),
        "r2"     : r2_score(y_test, y_pred),
        "cv_r2"  : cross_val_score(model, X, y, cv=5, scoring="r2").mean(),
    }
    print(f"Trained -> {name}")

## 8. Model Evaluation

In [ ]:
res_df = pd.DataFrame({
    "Model" : list(results.keys()),
    "R2"    : [r["r2"]    for r in results.values()],
    "RMSE"  : [r["rmse"]  for r in results.values()],
    "MAE"   : [r["mae"]   for r in results.values()],
    "CV R2" : [r["cv_r2"] for r in results.values()],
}).sort_values("R2", ascending=False).reset_index(drop=True)

best_name = res_df.iloc[0]["Model"]
print(f"Best Model: {best_name}")
res_df.round(4)

In [ ]:
rf = results["Random Forest"]["model"]
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature Importances (Random Forest):")
print(fi.round(4))

## 9. Visualisations

In [ ]:
best = results[best_name]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
BLUE, ORANGE, GREEN, RED = '#4C72B0', '#DD8452', '#55A868', '#C44E52'

# Model R2 comparison
ax = axes[0, 0]
colors = [RED if m == best_name else BLUE for m in res_df['Model']]
b = ax.bar(res_df['Model'], res_df['R2'], color=colors, edgecolor='white')
[ax.text(x.get_x()+x.get_width()/2, x.get_height()+0.005, f'{v:.3f}',
         ha='center', fontsize=8) for x, v in zip(b, res_df['R2'])]
ax.set_xticklabels(res_df['Model'], rotation=28, ha='right', fontsize=8)
ax.set_title('Model R2 Comparison', fontweight='bold')
ax.set_ylim(0, 1.08)

# Actual vs Predicted
ax = axes[0, 1]
ax.scatter(y_test, best['y_pred'], alpha=0.7, color=BLUE, s=55)
lims = [min(y_test.min(), best['y_pred'].min())-1,
        max(y_test.max(), best['y_pred'].max())+1]
ax.plot(lims, lims, '--', color=RED, lw=1.8)
ax.set_title(f'Actual vs Predicted', fontweight='bold')
ax.text(0.05, 0.9, f'R2={best["r2"]:.4f}', transform=ax.transAxes,
        color=RED, fontweight='bold')

# Residuals
ax = axes[0, 2]
res = y_test.values - best['y_pred']
ax.scatter(best['y_pred'], res, alpha=0.6, color=ORANGE, s=50)
ax.axhline(0, color=RED, linestyle='--', lw=1.5)
ax.set_title('Residual Plot', fontweight='bold')

# Feature importances
ax = axes[1, 0]
ax.barh(fi.index[::-1], fi.values[::-1], color=[BLUE, ORANGE, GREEN][::-1])
ax.set_title('Feature Importances (RF)', fontweight='bold')

# TV vs Sales
ax = axes[1, 1]
ax.scatter(df['TV'], df['Sales'], alpha=0.45, color=BLUE, s=40)
m, b_val = np.polyfit(df['TV'], df['Sales'], 1)
xs = np.linspace(df['TV'].min(), df['TV'].max(), 200)
ax.plot(xs, m*xs+b_val, color=RED, lw=2)
ax.set_title('TV Spend vs Sales', fontweight='bold')

# Sales distribution
ax = axes[1, 2]
ax.hist(df['Sales'], bins=25, color=GREEN, edgecolor='white', alpha=0.85)
ax.axvline(df['Sales'].mean(), color=RED, linestyle='--', lw=2)
ax.set_title('Sales Distribution', fontweight='bold')

plt.suptitle('Sales Prediction — Analysis Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
import os; os.makedirs('images', exist_ok=True)
plt.savefig('images/dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved images/dashboard.png")


## 10. Business Insights & Scenario Analysis

In [ ]:
lr = results["Linear Regression"]["model"]

scenarios = {
    "Low Budget"      : [50,  10,  10],
    "TV-Heavy"        : [250, 20,  20],
    "Radio-Heavy"     : [100, 80,  20],
    "Balanced"        : [150, 50,  50],
    "Digital-Focused" : [80,  90,  60],
    "Max Spend"       : [300, 100, 80],
}

print(f"{"Strategy":<18} {"TV":>6} {"Radio":>7} {"News":>6}  Predicted Sales")
print("-" * 58)
for label, vals in scenarios.items():
    pred = lr.predict([vals])[0]
    print(f"{label:<18} {vals[0]:>6}  {vals[1]:>6}  {vals[2]:>6}  ->  {pred:.2f}")

In [ ]:
coef = pd.Series(lr.coef_, index=FEATURES)
print("Linear Model Coefficients (sales units per $000 spend):")
for feat, c in coef.items():
    print(f"  {feat:<12}: {c:+.4f}")
print(f"  Intercept   : {lr.intercept_:.4f}")

## 11. Conclusion

### Key Findings

| Finding | Detail |
|---|---|
| **Best Model** | Polynomial Regression (degree 2) — R² = 0.987 |
| **Top Feature** | TV advertising (62% importance) |
| **Low Impact** | Newspaper (<2% importance) |
| **Best Strategy** | Max Spend → ~35.5 predicted sales units |

### Business Recommendations

1. **Prioritise TV** — highest ROI per dollar spent
2. **Leverage Radio** — strong synergy with TV
3. **Reduce Newspaper** — reallocate budget to TV or Radio
4. **Digital Strategy** works well at moderate budgets
5. Combined TV + Radio produces amplified (non-linear) returns

---
*CodeAlpha Internship Task 4 — Sales Prediction using Python*